# 03 — Scatter e histogramas

Scatter para relaciones entre dos variables numéricas. Histograma para ver cómo se distribuye una variable.

## Setup

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT  = find_project_root()
TRAIN = ROOT / 'data' / 'external' / 'train.csv'
AIR   = ROOT / 'data' / 'external' / 'Listings.csv'

air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)
air_clean = air[(air['price'] > 0) & (air['price'] < 500)].dropna(
    subset=['price', 'bedrooms', 'review_scores_rating', 'accommodates']
)
df = pd.read_csv(TRAIN, low_memory=False)
print(f'Airbnb filtrado: {air_clean.shape}')


## Histograma — distribución de precio

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(
    air_clean['price'],
    bins=60,
    color='steelblue',
    edgecolor='white',
    linewidth=0.5,
)

# Línea de media y mediana
media   = air_clean['price'].mean()
mediana = air_clean['price'].median()

ax.axvline(media,   color='tomato',     linestyle='--', linewidth=1.5, label=f'Media: ${media:.0f}')
ax.axvline(mediana, color='darkorange', linestyle=':',  linewidth=1.5, label=f'Mediana: ${mediana:.0f}')

ax.set_title('Distribución de precio por noche — Airbnb')
ax.set_xlabel('Precio (USD/noche)')
ax.set_ylabel('Frecuencia')
ax.legend()
ax.grid(axis='y', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()


## Histogramas superpuestos — comparar distribuciones

In [ ]:
# Precio por tipo de habitación
tipos = air_clean['room_type'].value_counts().head(3).index
colores_tipo = ['steelblue', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(9, 4))

for tipo, color in zip(tipos, colores_tipo):
    datos = air_clean[air_clean['room_type'] == tipo]['price']
    ax.hist(datos, bins=40, alpha=0.55, color=color, edgecolor='white',
            linewidth=0.3, label=tipo)

ax.set_title('Distribución de precio por tipo de habitación')
ax.set_xlabel('Precio (USD/noche)')
ax.set_ylabel('Frecuencia')
ax.legend()
ax.grid(axis='y', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()


## Scatter — relación entre precio y accommodates

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    air_clean['accommodates'],
    air_clean['price'],
    alpha=0.25,       # transparencia para ver densidad
    s=15,             # tamaño de cada punto
    color='steelblue',
    edgecolors='none',
)

ax.set_title('Precio vs capacidad de alojados')
ax.set_xlabel('Número de personas que caben')
ax.set_ylabel('Precio por noche (USD)')
ax.grid(linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()


## Scatter con color como tercera dimensión — coloreado por room_type

In [ ]:
mapa_color = {
    'Entire home/apt':  'steelblue',
    'Private room':     'darkorange',
    'Shared room':      'seagreen',
    'Hotel room':       'orchid',
}

fig, ax = plt.subplots(figsize=(9, 5))

for tipo, color in mapa_color.items():
    sub = air_clean[air_clean['room_type'] == tipo]
    ax.scatter(
        sub['accommodates'],
        sub['price'],
        alpha=0.3,
        s=12,
        color=color,
        edgecolors='none',
        label=tipo,
    )

ax.set_title('Precio vs accommodates por tipo de habitación')
ax.set_xlabel('Accommodates')
ax.set_ylabel('Precio por noche (USD)')
ax.legend(markerscale=2, fontsize=9)
ax.grid(linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()


## Scatter con línea de tendencia

In [ ]:
from numpy.polynomial import polynomial as P

x = air_clean['accommodates'].values
y = air_clean['price'].values

# Ajuste lineal
coef = np.polyfit(x, y, 1)
x_linea = np.linspace(x.min(), x.max(), 100)
y_linea = np.polyval(coef, x_linea)

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(x, y, alpha=0.2, s=12, color='steelblue', edgecolors='none',
           label='datos')
ax.plot(x_linea, y_linea, color='tomato', linewidth=2,
        label=f'tendencia: ${coef[0]:.1f}/persona + ${coef[1]:.0f}')

ax.set_title('Precio vs accommodates con línea de tendencia')
ax.set_xlabel('Accommodates')
ax.set_ylabel('Precio (USD)')
ax.legend()
ax.grid(linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()


---
## Resumen

| Tipo | Función | Parámetros clave |
|------|---------|------------------|
| Histograma | `ax.hist(data, bins=N)` | `alpha`, `edgecolor`, `density=True` para proporción |
| Superpuesto | múltiples `ax.hist()` | `alpha=0.5` para ver solapamiento |
| Scatter | `ax.scatter(x, y)` | `alpha`, `s` (tamaño), `c` (color por valor) |
| Línea referencia | `ax.axvline(x)` / `ax.axhline(y)` | `linestyle`, `color`, `label` |
| Tendencia | `np.polyfit` + `ax.plot` | fácil para lineal, usar scipy para más control |
